# Reading The Data
***

In [1]:
import pandas as pd
data = pd.read_csv('../Data/healthcare-dataset-stroke-data.csv')

# Cleaning The Data
***

### Remove Unnecessary Columns

The `id` column was removed because it is only a unique identifier for each patient and does not contain any information relevant to stroke prediction. Therefore, it was excluded from the dataset before model development.

In [2]:
# Drop the 'id' column 
data.drop(columns=['id'], inplace=True)

# Verify the changes
data.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

### DEALING WITH MISSING VALUES
The bmi attribute contained 201 missing values (approximately 3.9% of the dataset). Instead of removing the affected records, we chose to impute the missing values using the median in order to preserve all observations and avoid unnecessary data loss. Since the bmi feature contains outliers and exhibits a slightly skewed distribution, the median was preferred over the mean because it is less affected by extreme values. This approach helps maintain the overall distribution of the data while ensuring that the dataset remains complete for subsequent analysis and model training.

In [3]:
# Check median BMI
median_bmi = data['bmi'].median()
print("Median BMI:", median_bmi)

# Fill missing BMI values
data['bmi'] = data['bmi'].fillna(median_bmi)

# Verify no missing values remain
data.isna().sum()

Median BMI: 28.1


gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

### DEALING WITH OUTLIERS
Outlier analysis revealed extreme values in the avg_glucose_level and bmi features, which are common in healthcare datasets due to natural variations among patients. Binary variables such as hypertension and heart_disease were excluded from outlier analysis because they contain only binary values (0 and 1), making IQR-based outlier detection unsuitable for these features.

To reduce the influence of extreme values while preserving clinically meaningful observations, the continuous features were treated using IQR-based capping.

Values exceeding the calculated upper or lower bounds were replaced with the corresponding threshold values.

This approach minimizes the impact of extreme observations while maintaining the overall integrity and distribution of the dataset, providing a more robust foundation for subsequent modeling.

In [4]:
# Check the extreme values for glucose and BMI
print("Glucose Range:", data['avg_glucose_level'].min(), "to", data['avg_glucose_level'].max())
print("BMI Range:", data['bmi'].min(), "to", data['bmi'].max())

Glucose Range: 55.12 to 271.74
BMI Range: 10.3 to 97.6


In [5]:
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower_bound, upper_bound)

# Apply capping only to relevant columns
cap_outliers(data, 'avg_glucose_level')
cap_outliers(data, 'bmi')

# Verify
data[['avg_glucose_level', 'bmi']].describe()

,avg_glucose_level,bmi
count,5110.000000,5110.000000
mean,100.996204,28.690411
std,33.214738,7.120858
min,55.120000,10.300000
25%,77.245000,23.800000
50%,91.885000,28.100000
75%,114.090000,32.800000
max,169.357500,46.300000


### STANDARDIZATION
To ensure comparability among features and improve model performance, all continuous variables — age, average glucose level, and BMI — were standardized using the Z-score normalization technique. This transformation rescales values to have a mean of 0 and a standard deviation of 1, ensuring that each variable contributes equally to model training.

Standardization was chosen over simple normalization because the distributions of these health-related features are approximately bell-shaped and contain outliers, which are more effectively managed under a standardized scale. This process enhances numerical stability and prepares the dataset for reliable and efficient learning in subsequent modeling stages.

In [6]:
from sklearn.preprocessing import StandardScaler

# Select numeric columns to scale
numeric_features = ['age', 'avg_glucose_level', 'bmi']

# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the data
data[numeric_features] = scaler.fit_transform(data[numeric_features])

# Verify results
data[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
age,5110.0,5.005781e-17,1.000098,-1.908261,-0.806115,0.078432,0.786070,1.714845
avg_glucose_level,5110.0,-3.392807e-16,1.000098,-1.381335,-0.715150,-0.274339,0.394255,2.058363
bmi,5110.0,-7.230572e-17,1.000098,-2.582864,-0.686840,-0.082921,0.577176,2.473201


In [7]:

import joblib

# Save the scaler
joblib.dump(
    scaler,
    r"D:\DEPI_Round 4\Graduation Project\Stroke-Risk-Prediction\Models\scaler.pkl"
)

['D:\\DEPI_Round 4\\Graduation Project\\Stroke-Risk-Prediction\\Models\\scaler.pkl']

# EXPORT THE CLEANED DATASET
***

In [7]:
data.to_csv('../Data/stroke_data_cleaned.csv', index=False)